In [ ]:
import ast
import os
import pandas as pd

In [ ]:
class CleanAndNormalizeAST(ast.NodeTransformer):
    """
    This class safely traverses the parsed tree. 
    It removes docstrings and anonymizes variable/function names.
    """
    def __init__(self):
        self.id_map = {}
        self.counter = 0

    def visit_Expr(self, node):
        # 1. Safely remove Docstrings (standalone string expressions)
        # In Python 3.8+, strings are represented as ast.Constant
        if isinstance(node.value, ast.Constant) and isinstance(node.value.value, str):
            return None # Returning None deletes this node from the structural tree entirely
        return self.generic_visit(node)

    def visit_Name(self, node):
        # 2. Normalize variable names to var_1, var_2, etc.
        if isinstance(node.ctx, (ast.Store, ast.Load)):
            if node.id not in self.id_map:
                self.counter += 1
                self.id_map[node.id] = f"var_{self.counter}"
            node.id = self.id_map[node.id]
        return self.generic_visit(node)

    def visit_FunctionDef(self, node):
        # 3. Normalize defined function names
        if node.name not in self.id_map:
            self.counter += 1
            self.id_map[node.name] = f"func_{self.counter}"
        node.name = self.id_map[node.name]
        return self.generic_visit(node)

In [ ]:
def process_file_to_ast(file_path):
    """
    Reads the raw file, parses it (ignoring comments natively), 
    cleans docstrings, and normalizes identifiers.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            source_code = f.read()
            
        # Parse the raw file directly (automatically ignores comments and whitespace)
        raw_tree = ast.parse(source_code)
        
        # Apply our cleaner and normalizer
        transformer = CleanAndNormalizeAST()
        clean_tree = transformer.visit(raw_tree)
        
        # Fix line numbers and locations after modifying the tree
        ast.fix_missing_locations(clean_tree)
        
        return ast.dump(clean_tree, annotate_fields=False)
    except Exception as e:
        print(f"   [!] Error parsing {os.path.basename(file_path)}: {e}")
        return None

In [ ]:
def build_final_dataset(labels_csv_path, raw_submissions_dir, output_csv):
    print("[*] Loading labeled dataset...")
    df = pd.read_csv("Data/cheatind_dataset.csv")
    
    ast_cache = {}
    def get_cached_ast(file_name):
        if file_name not in ast_cache:
            full_path = os.path.join("Data/original-codes/", file_name)
            ast_cache[file_name] = process_file_to_ast(full_path)
        return ast_cache[file_name]
        
    print("[*] Parsing raw code into anonymized AST strings...")
    df['AST_1'] = df['File_1'].apply(get_cached_ast)
    df['AST_2'] = df['File_2'].apply(get_cached_ast)
    
    # Drop rows where actual Python structural syntax errors occurred
    initial_len = len(df)
    df_cleaned = df.dropna(subset=['AST_1', 'AST_2']).reset_index(drop=True)
    
    df_cleaned.to_csv("Data/ast_dataset.csv", index=False)
    print(f"[+] Success! {len(df_cleaned)}/{initial_len} pairs saved to Data/ast_dataset.csv.")

In [ ]:
if __name__ == "__main__":
    build_final_dataset()

Processing cleaned code files into anonymized AST structures...
Error processing submission112.py: invalid syntax (<unknown>, line 14)
Error processing submission113.py: invalid syntax (<unknown>, line 14)
Error processing submission114.py: invalid syntax (<unknown>, line 14)
Error processing submission115.py: invalid syntax (<unknown>, line 14)
Error processing submission116.py: invalid syntax (<unknown>, line 24)
Error processing submission158.py: invalid syntax (<unknown>, line 14)
[+] Successfully structured 282/293 pairs.
[+] Dataset with paired AST structures saved to: Data/ast_dataset.csv
